# 35. 합성 평가셋 — 기준선 측정

사전 등록: `docs/plans/NLR_EVALSET_PREREGISTRATION.md`
입력: 노트북 34 의 구조화 체크포인트 600건

구조화 결과로 검색해서 **하드 AND 와 소프트 점수를 나란히 채점한다.**
`spec.md` §7.3 의 절개 방식(baseline → 사전 → 정규화)을 따른다.

## 0. 실행 조건과 한계

- **API 호출 0회.** 노트북 34 의 체크포인트를 읽는다
- **accord 만으로 검색한다.** 조건 집합 C 가 accord 기준이므로 note 로 검색하면 채점이 어긋난다.
  note 값은 개수만 기록한다
- **조건 완화를 넣지 않는다.** `spec.md` §3 ③ 의 5단계 완화는 변수를 하나 더 만든다.
  결과 3개 미만 건수를 진단 지표로만 센다
- **`avoid` 는 이름 동일성만 쓴다.** §8 11번이 미정이라 `spec.md` §3 예제의 가정을 따른다.
  사전 항목 전체를 쓰는 쪽은 이 노트북에서 재지 않는다
- 평가 데이터·사전·원자료는 읽기만 한다

In [1]:
import hashlib
import json
import pathlib
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 260)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

TOP_K = 5          # 상위 몇 개를 채점하는가
TOP_K_C = 3        # 조건 집합 C 의 크기 (사전 등록 §7)
MIN_RESULTS = 3    # 이 미만이면 '결과 부족' 으로 센다 (완화는 하지 않는다)

print("REPORT_ONLY:", REPORT_ONLY)

REPORT_ONLY: False


## 1. 경로 · 입력 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
KNOW_DIR = PROJECT_ROOT / "data" / "scent_knowledge"

INPUT_PATHS = {
    "checkpoint": OUTPUT_DIR / "34_evalset_stage1_checkpoint.csv",
    "answer_key": OUTPUT_DIR / "32_evalset_answer_key.csv",
    "lexicon": KNOW_DIR / "domain_lexicon_v1_1.csv",
    "accord_dict": OUTPUT_DIR / "10_accord_dictionary.csv",
    "note_dict": OUTPUT_DIR / "10_note_dictionary.csv",
    "perfumes_csv": PROJECT_ROOT / "perfumes.csv",
}
OUTPUT_PATHS = {
    "scores": OUTPUT_DIR / "35_baseline_scores.csv",
    "per_query": OUTPUT_DIR / "35_baseline_per_query.csv",
    "layers": OUTPUT_DIR / "35_normalization_layers.csv",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    d = hashlib.sha256()
    with pathlib.Path(path).open("rb") as h:
        for chunk in iter(lambda: h.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
PROTECTED = {p.resolve() for p in INPUT_PATHS.values()}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
checkpoint,0fb2aeba10b84a1b
answer_key,d81e0f4fb2f1e1cd
lexicon,9e0a1365437a27a0
accord_dict,567e91370e575731
note_dict,72de5566687add52
perfumes_csv,cec1ea0b49885303


## 2. 사전 등록 — 채점하기 전에 고정한다

**정규화 층을 결과를 보기 전에 정한다.** 노트북 25 가 채점 규칙을 층으로 나눠 사전 등록하고
층별 기여를 보고한 방식을 그대로 쓴다. 결과를 보고 층을 추가하면 사전 등록이 깨진다.

In [3]:
PREREG = {
    "조건 1 (목록 안만)": "LLM 이 뽑은 향 이름 중 accord 92개 안에 있는 것만 쓴다. "
                          "목록 밖은 버리고 개수를 지표로 남긴다 (spec §5.2)",
    "조건 2 (+ 사전)": "조건 1 + Domain Lexicon v1.1 조회. "
                       "additional_requirements 와 원문에서 사전 표현을 찾아 accord 를 더한다",
    "조건 3 (+ 정규화)": "조건 2 + 아래 정규화 층 L1~L4",
    "정규화 L0": "원문 그대로 대조",
    "정규화 L1": "소문자 · 앞뒤 공백 · 연속 공백 정리",
    "정규화 L2": "복수형 — 끝 s 제거 후 재대조 (white florals → white floral)",
    "정규화 L3": "형용사화 — 끝 e 제거 후 y 부착, 또는 y 부착 (fruit → fruity, wood → woody)",
    "정규화 L4": "note 목록까지 확장 — accord 가 아니면 note 로 인정 (검색에는 쓰지 않고 기록만)",
    "검색 A (하드 AND)": "core accord 를 전부 가진 향수만. strength 합으로 정렬",
    "검색 B (소프트)": "필터 없음. 노트북 11 방식 — 쿼리 accord strength 의 평균",
    "avoid": "이름 동일성만. 사전 항목 전체를 쓰는 쪽은 재지 않는다 (§8 11번 미정)",
    "조건 완화": "하지 않는다. 결과 3개 미만 건수만 진단으로 센다",
    "동점 처리": "strength 합 내림차순, 같으면 perfume id 오름차순 (결정적)",
    "주 지표": f"Condition Precision@{TOP_K} — 상위 {TOP_K}개 중 C 를 전부 가진 수 / {TOP_K}",
    "보조 지표": "느슨한 정의(C 중 2개 이상) · 원본 Recall@5 · MRR · 갈래 A−B 차이",
}
display(pd.Series(PREREG, name="사전 등록").to_frame())

,사전 등록
조건 1 (목록 안만),LLM 이 뽑은 향 이름 중 accord 92개 안에 있는 것만 쓴다. 목록 밖은 버리고 개수를 지표...
조건 2 (+ 사전),조건 1 + Domain Lexicon v1.1 조회. additional_requirements 와...
조건 3 (+ 정규화),조건 2 + 아래 정규화 층 L1~L4
정규화 L0,원문 그대로 대조
정규화 L1,소문자 · 앞뒤 공백 · 연속 공백 정리
정규화 L2,복수형 — 끝 s 제거 후 재대조 (white florals → white floral)
정규화 L3,"형용사화 — 끝 e 제거 후 y 부착, 또는 y 부착 (fruit → fruity, wood → wo..."
정규화 L4,note 목록까지 확장 — accord 가 아니면 note 로 인정 (검색에는 쓰지 않고 기록만)
검색 A (하드 AND),core accord 를 전부 가진 향수만. strength 합으로 정렬
검색 B (소프트),필터 없음. 노트북 11 방식 — 쿼리 accord strength 의 평균


## 3. 데이터 로드 · 재현 게이트

In [4]:
ck = pd.read_csv(INPUT_PATHS["checkpoint"])
key = pd.read_csv(INPUT_PATHS["answer_key"]).set_index("perfume_id")
lex = pd.read_csv(INPUT_PATHS["lexicon"], keep_default_na=False, dtype=str)
accord_dict = pd.read_csv(INPUT_PATHS["accord_dict"])
note_names = set(pd.read_csv(INPUT_PATHS["note_dict"]).iloc[:, 0].astype(str).str.lower())

perf = pd.read_csv(INPUT_PATHS["perfumes_csv"], usecols=["id", "accords"], low_memory=False)
print(f"향수 {len(perf):,} / 구조화 {len(ck)} / 정답키 {len(key)} / 사전 {len(lex)}행")


def parse_accords(value):
    """'name:strength|...' → [(name, strength)] 내림차순. list[tuple[str, float]]."""
    out = []
    for part in str(value).split("|"):
        if not part or part == "nan":
            continue
        n, _, s = part.partition(":")
        try:
            out.append((n, float(s)))
        except ValueError:
            continue
    out.sort(key=lambda x: (-x[1], x[0]))
    return out


pairs = perf["accords"].map(parse_accords)
ACCORDS = sorted(accord_dict["accord"])
AIDX = {a: i for i, a in enumerate(ACCORDS)}

counts = {}
for ps in pairs:
    for n, _ in ps:
        counts[n] = counts.get(n, 0) + 1
gate_err = sum(counts.get(r["accord"], 0) != r["perfume_count"]
               for r in accord_dict.to_dict("records"))
print(f"[게이트] accord {len(ACCORDS)}개 perfume_count 오차 {gate_err}건")
if gate_err:
    raise RuntimeError("재현 게이트 실패")

향수 131,930 / 구조화 600 / 정답키 100 / 사전 44행


[게이트] accord 92개 perfume_count 오차 0건


### 강도 행렬

131,930 × 92 float32 는 약 48MB 다. 한 번 만들어 두면 검색이 행렬 연산으로 끝난다.

In [5]:
M = np.zeros((len(perf), len(ACCORDS)), dtype=np.float32)
for r, ps in enumerate(pairs):
    for n, s in ps:
        j = AIDX.get(n)
        if j is not None:
            M[r, j] = s
HAS = M > 0
PID = perf["id"].to_numpy()
print(f"강도 행렬 {M.shape} / {M.nbytes/1e6:.0f}MB / 비영 {int(HAS.sum()):,}")

강도 행렬 (131930, 92) / 49MB / 비영 1,000,570

## 4. 정규화 층 — 사전 등록한 L0~L4

In [6]:
ACC_SET = set(ACCORDS)


def norm_layers(raw):
    """향 이름 하나를 층별로 해석한다. (accord 또는 None, 층 이름)."""
    s0 = str(raw)
    if s0 in ACC_SET:
        return s0, "L0"
    s1 = re.sub(r"\s+", " ", s0.strip().lower())
    if s1 in ACC_SET:
        return s1, "L1"
    s2 = re.sub(r"s\b", "", s1).strip()          # 복수형
    if s2 in ACC_SET:
        return s2, "L2"
    for cand in (re.sub(r"e$", "", s2) + "y", s2 + "y"):   # 형용사화
        if cand in ACC_SET:
            return cand, "L3"
    if s1 in note_names or s2 in note_names:
        return None, "L4_note"
    return None, "밖"


rows = []
for r in ck.to_dict("records"):
    o = json.loads(r["parsed"]) if isinstance(r["parsed"], str) and r["parsed"].strip() else {}
    for v in (o.get("scent_preference") or []):
        acc, layer = norm_layers(v)
        rows.append({"arm": r["arm"], "원문": v, "accord": acc, "층": layer})
layer_df = pd.DataFrame(rows)
display(pd.crosstab(layer_df["층"], layer_df["arm"], margins=True))
print("\n층별로 새로 건진 accord 값 (L0 는 원래 맞던 것)")
display(layer_df[layer_df["층"].isin(["L2", "L3"])]
        .groupby(["층", "원문"]).size().sort_values(ascending=False).head(12).to_frame("건수"))

arm,A,B,All
층,,,
L0,400,157,557
L1,182,110,292
L2,20,2,22
L3,34,13,47
L4_note,43,70,113
밖,83,60,143
All,762,412,1174



층별로 새로 건진 accord 값 (L0 는 원래 맞던 것)


건수
층  원문               
L3 fruit          28
L2 spices         14
   white florals   7
L3 Fruit           5
   warm spices     4
   wood            3
   soap            2
   Fruits          1
L2 Spices          1
L3 Wood            1
   Warm Spices     1
   Powder          1

## 5. 사전 조회

In [7]:
lex_acc = lex[(lex["candidate_type"] == "ACCORD") &
              (lex["target_field"] != "NO_MAPPING")].copy()
forms = {}
for r in lex.to_dict("records"):
    for f in [r["expression"]] + [a for a in r["aliases"].split("|") if a]:
        if f:
            forms.setdefault(f, set()).add(r["expression"])


def lexicon_lookup(text):
    """문장에서 사전 표현을 찾아 core accord 집합을 낸다. (set, list[str])."""
    hit_expr = set()
    for form, exprs in forms.items():
        if form and form in text:
            hit_expr |= exprs
    core = set()
    for e in hit_expr:
        g = lex_acc[lex_acc["expression"] == e]
        for r in g.to_dict("records"):
            cond = r["match_condition"]
            if cond.startswith("query_contains:"):
                toks = [t for t in cond.split(":", 1)[1].split(",") if t]
                if not any(t in text for t in toks):
                    continue
            if r["required"] == "core":
                core.add(r["candidate_name"])
    return core & ACC_SET, sorted(hit_expr)

## 6. 쿼리별 accord 집합 — 조건 1 · 2 · 3

In [8]:
def build_targets(rec):
    """구조화 결과에서 조건별 accord 집합과 avoid 를 만든다. dict."""
    o = json.loads(rec["parsed"]) if isinstance(rec["parsed"], str) and rec["parsed"].strip() else {}
    scent = [str(v) for v in (o.get("scent_preference") or [])]
    add = [str(v) for v in (o.get("additional_requirements") or [])]
    avoid_raw = [str(v) for v in (o.get("avoid") or [])]

    c1 = {v for v in scent if v in ACC_SET or v.strip().lower() in ACC_SET}
    c1 = {v if v in ACC_SET else v.strip().lower() for v in c1}

    text = " ".join([str(rec["sentence"])] + add)
    lex_core, hit_expr = lexicon_lookup(text)
    c2 = c1 | lex_core

    c3 = set(c2)
    n_note, n_out = 0, 0
    for v in scent:
        acc, layer = norm_layers(v)
        if acc:
            c3.add(acc)
        elif layer == "L4_note":
            n_note += 1
        else:
            n_out += 1

    # avoid — 이름 동일성만
    av = set()
    for v in avoid_raw:
        acc, _ = norm_layers(v)
        if acc:
            av.add(acc)
    return {"c1": sorted(c1), "c2": sorted(c2), "c3": sorted(c3),
            "avoid": sorted(av), "lex_expr": hit_expr,
            "n_note": n_note, "n_outside": n_out}


targets = [build_targets(r) for r in ck.to_dict("records")]
tdf = pd.DataFrame(targets)
tdf["arm"] = ck["arm"].values
for c in ("c1", "c2", "c3"):
    tdf[f"{c}_n"] = tdf[c].map(len)
display(tdf.groupby("arm")[["c1_n", "c2_n", "c3_n", "n_note", "n_outside"]].mean().round(2))
print("\naccord 가 0개인 쿼리")
display(pd.DataFrame({c: tdf.groupby("arm")[f"{c}_n"].apply(lambda s: int((s == 0).sum()))
                      for c in ("c1", "c2", "c3")}))
print("\naccord 가 2개 이상인 쿼리 (하드 AND 가 성립하는 최소 조건)")
display(pd.DataFrame({c: tdf.groupby("arm")[f"{c}_n"].apply(lambda s: int((s >= 2).sum()))
                      for c in ("c1", "c2", "c3")}))

,c1_n,c2_n,c3_n,n_note,n_outside
arm,,,,,
A,1.93,2.52,2.70,0.14,0.28
B,0.89,1.42,1.47,0.23,0.20



accord 가 0개인 쿼리


,c1,c2,c3
arm,,,
A,39,36,35
B,135,101,100



accord 가 2개 이상인 쿼리 (하드 AND 가 성립하는 최소 조건)


,c1,c2,c3
arm,,,
A,207,224,239
B,80,141,145


## 7. 검색 — 하드 AND 와 소프트 점수

두 방식 모두 같은 accord 집합을 받아 상위 5개를 낸다. 동점은 perfume id 오름차순으로 자른다.

In [9]:
def search(acc_list, avoid_list, mode):
    """상위 TOP_K 향수의 행 인덱스. (np.ndarray, 후보 수, 5위 동점 수)."""
    cols = [AIDX[a] for a in acc_list if a in AIDX]
    if not cols:
        return np.array([], dtype=int), 0, 0
    mask = np.ones(len(perf), dtype=bool)
    for a in avoid_list:
        j = AIDX.get(a)
        if j is not None:
            mask &= ~HAS[:, j]
    if mode == "hard":
        for j in cols:
            mask &= HAS[:, j]
        score = M[:, cols].sum(axis=1)
    else:
        score = M[:, cols].mean(axis=1)
        mask &= score > 0
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return np.array([], dtype=int), 0, 0
    sc = score[idx]
    order = np.lexsort((PID[idx], -sc))
    idx = idx[order]
    tie = int((sc == sc[order][min(TOP_K, len(idx)) - 1]).sum()) if len(idx) >= TOP_K else len(idx)
    return idx[:TOP_K], int(idx.size), tie

## 8. 채점

In [10]:
def grade(top_idx, pid_true, C):
    """상위 결과를 정답 조건에 대조한다. dict."""
    if top_idx.size == 0:
        return {"P@5_strict": 0.0, "P@5_loose": 0.0, "hit": 0, "rr": 0.0}
    cols = [AIDX[a] for a in C if a in AIDX]
    sub = HAS[np.ix_(top_idx, cols)]
    strict = sub.all(axis=1).mean()
    loose = (sub.sum(axis=1) >= 2).mean()
    hits = np.flatnonzero(PID[top_idx] == pid_true)
    return {"P@5_strict": float(strict), "P@5_loose": float(loose),
            "hit": int(hits.size > 0),
            "rr": float(1.0 / (hits[0] + 1)) if hits.size else 0.0}


records = []
for rec, tg in zip(ck.to_dict("records"), targets):
    pid = rec["perfume_id"]
    C = str(key.loc[pid, "C"]).split("|")
    for cond in ("c1", "c2", "c3"):
        for mode in ("hard", "soft"):
            top, ncand, tie = search(tg[cond], tg["avoid"], mode)
            g = grade(top, pid, C)
            records.append({
                "query_id": rec["query_id"], "arm": rec["arm"], "perfume_id": pid,
                "조건": cond, "검색": mode, "target_n": len(tg[cond]),
                "후보 수": ncand, "5위 동점": tie,
                "결과 부족": int(ncand < MIN_RESULTS),
                "검색 불가": int(ncand == 0),
                "random_expect": float(key.loc[pid, "random_expect_at5"]), **g,
            })
per_query = pd.DataFrame(records)
print(f"채점 {len(per_query):,}행 = 600문장 × 조건 3 × 검색 2")

채점 3,600행 = 600문장 × 조건 3 × 검색 2


## 9. 결과

In [11]:
agg = (per_query.groupby(["조건", "검색", "arm"])
       .agg(**{"P@5 엄격": ("P@5_strict", "mean"),
               "P@5 느슨": ("P@5_loose", "mean"),
               "원본 Recall@5": ("hit", "mean"),
               "MRR": ("rr", "mean"),
               "랜덤 기대": ("random_expect", "mean"),
               "검색 불가": ("검색 불가", "mean"),
               "결과 부족": ("결과 부족", "mean"),
               "5위 동점 중앙": ("5위 동점", "median"),
               "target 평균": ("target_n", "mean")})
       .round(4).reset_index())
display(agg)

,조건,검색,arm,P@5 엄격,P@5 느슨,원본 Recall@5,MRR,랜덤 기대,검색 불가,결과 부족,5위 동점 중앙,target 평균
0,c1,hard,A,0.3023,0.6890,0.0033,0.0017,0.0133,0.1333,0.1333,19.0,1.9333
1,c1,hard,B,0.1453,0.3837,0.0033,0.0033,0.0133,0.4567,0.4633,2.0,0.8900
2,c1,soft,A,0.3013,0.6907,0.0033,0.0017,0.0133,0.1300,0.1300,20.0,1.9333
3,c1,soft,B,0.1460,0.3893,0.0033,0.0033,0.0133,0.4500,0.4500,2.0,0.8900
4,c2,hard,A,0.2928,0.6806,0.0000,0.0000,0.0133,0.1400,0.1433,2.0,2.5167
5,c2,hard,B,0.1469,0.4139,0.0033,0.0033,0.0133,0.3533,0.3567,2.0,1.4200
6,c2,soft,A,0.2813,0.6840,0.0000,0.0000,0.0133,0.1200,0.1200,2.0,2.5167
7,c2,soft,B,0.1467,0.4227,0.0033,0.0033,0.0133,0.3367,0.3367,2.0,1.4200
8,c3,hard,A,0.3048,0.6613,0.0033,0.0007,0.0133,0.1767,0.1867,2.0,2.6967
9,c3,hard,B,0.1513,0.4190,0.0033,0.0033,0.0133,0.3567,0.3633,2.0,1.4667


### 갈래 A − 갈래 B = 순환의 크기

갈래 A 는 accord 를 보고 만든 문장이라 되돌리기 쉽다. 그 차이가 순환이 점수를 부풀린 양이다.

In [12]:
gap = (agg.pivot_table(index=["조건", "검색"], columns="arm",
                       values=["P@5 엄격", "원본 Recall@5"])
       .round(4))
gap[("P@5 엄격", "A−B")] = gap[("P@5 엄격", "A")] - gap[("P@5 엄격", "B")]
gap[("원본 Recall@5", "A−B")] = gap[("원본 Recall@5", "A")] - gap[("원본 Recall@5", "B")]
display(gap)

P@5 엄격         원본 Recall@5          P@5 엄격 원본 Recall@5
arm           A       B           A       B     A−B         A−B
조건 검색                                                          
c1 hard  0.3023  0.1453      0.0033  0.0033  0.1570      0.0000
   soft  0.3013  0.1460      0.0033  0.0033  0.1553      0.0000
c2 hard  0.2928  0.1469      0.0000  0.0033  0.1459     -0.0033
   soft  0.2813  0.1467      0.0000  0.0033  0.1346     -0.0033
c3 hard  0.3048  0.1513      0.0033  0.0033  0.1535      0.0000
   soft  0.3107  0.1507      0.0033  0.0033  0.1600      0.0000

### 사전과 정규화가 각각 얼마나 기여했나

In [13]:
base = agg.set_index(["조건", "검색", "arm"])
contrib = []
for mode in ("hard", "soft"):
    for arm in ("A", "B"):
        v = {c: base.loc[(c, mode, arm), "P@5 엄격"] for c in ("c1", "c2", "c3")}
        contrib.append({"검색": mode, "arm": arm,
                        "조건1 목록 안만": v["c1"],
                        "조건2 +사전": v["c2"], "Δ 사전": round(v["c2"] - v["c1"], 4),
                        "조건3 +정규화": v["c3"], "Δ 정규화": round(v["c3"] - v["c2"], 4)})
display(pd.DataFrame(contrib))

,검색,arm,조건1 목록 안만,조건2 +사전,Δ 사전,조건3 +정규화,Δ 정규화
0,hard,A,0.3023,0.2928,-0.0095,0.3048,0.0120
1,hard,B,0.1453,0.1469,0.0016,0.1513,0.0044
2,soft,A,0.3013,0.2813,-0.0200,0.3107,0.0294
3,soft,B,0.1460,0.1467,0.0007,0.1507,0.0040


## 10. 저장

In [14]:
layers_out = layer_df.groupby(["arm", "층"]).size().reset_index(name="건수")
write_output(OUTPUT_PATHS["scores"], lambda p: agg.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["per_query"],
             lambda p: per_query.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["layers"],
             lambda p: layers_out.to_csv(p, index=False, encoding="utf-8-sig"))

저장: analysis_outputs\35_baseline_scores.csv
저장: analysis_outputs\35_baseline_per_query.csv
저장: analysis_outputs\35_normalization_layers.csv


WindowsPath('C:/Users/SSAFY/Desktop/hyanghae/EDA/analysis_outputs/35_normalization_layers.csv')

## 11. 가드 검증

In [15]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in after if after[k] != input_hashes_before[k]]
if changed:
    raise RuntimeError(f"입력 파일이 변경됐다: {changed}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))

입력 해시 불변 확인: checkpoint, answer_key, lexicon, accord_dict, note_dict, perfumes_csv
